In [31]:
import os, torch, numpy as np
import matplotlib.pyplot as plt
from pymatgen.core import Structure, Molecule
from pymatgen.io.xyz import XYZ
import wandb

In [2]:
target_graph_list = torch.load(
    "/jbod/rokubo/data/diffusion_model/wandb/run-20260303_170824-k2en1th3/files/original_graph.pt",
    weights_only=False
)
generated_graph_list = torch.load(
    "/jbod/rokubo/data/diffusion_model/wandb/run-20260303_170824-k2en1th3/files/generated_graph.pt",
    weights_only=False
)

In [59]:
def one_hot_to_atom_type(one_hot: torch.Tensor) -> torch.Tensor:
    atom_species_tensor = torch.ones(one_hot.shape[0], dtype=torch.long).to(one_hot.device)
    atom_species_tensor[one_hot[:, 0] == 1] = 8  # O
    atom_species_tensor[one_hot[:, 1] == 1] = 14  # Si
    return atom_species_tensor

def graph2molecule(graph):
    return Molecule(species=one_hot_to_atom_type(graph.x).cpu().numpy(), coords=graph.pos.cpu().numpy())

In [22]:
for _ in range(1):
    print(target_graph_list[_])
    print(target_graph_list[_].x)
    print(one_hot_to_atom_type(target_graph_list[_].x))

Data(x=[2, 2], pos=[2, 3], spectrum=[2, 200], spectrum_raw=[2, 200], excited_atom=[2, 1], id='mp-10064_1_minus_3', formula='SiO2')
tensor([[1, 0],
        [0, 1]], device='cuda:0')
tensor([ 8, 14], device='cuda:0')


In [23]:
Molecule(species=one_hot_to_atom_type(target_graph_list[0].x).cpu().numpy(), coords=target_graph_list[0].pos.cpu().numpy())

Molecule Summary
Site: O (0.0000, 0.0000, 0.0000)
Site: Si (-0.0022, -1.8022, -0.0037)

In [57]:
sorted_id_rmsd = torch.load(
    "/jbod/rokubo/data/diffusion_model/xyz/zbo1ivw1/sorted_id_rmsd.pt",
    weights_only=False
)
sorted_id = [id for id, _ in sorted_id_rmsd]
sorted_rmsd = [rmsd for _, rmsd in sorted_id_rmsd]
for i in range(len(sorted_id)):
    if "mp-10064" in sorted_id[i]:
        print(sorted_id[i], sorted_rmsd[i])
print(np.mean(sorted_rmsd))

0.063505038432961


In [60]:
for i in range(len(target_graph_list)):
    try:
        target_graph = target_graph_list[i]
        generated_graph = generated_graph_list[i][-1]
        mol_id = target_graph.id
        os.makedirs(f"/jbod/rokubo/data/diffusion_model/xyz/k2en1th3/{mol_id}", exist_ok=True)
        target_molecule = graph2molecule(target_graph)
        generated_molecule = graph2molecule(generated_graph)
        counter = 0
        while os.path.exists(f"/jbod/rokubo/data/diffusion_model/xyz/k2en1th3/{mol_id}/{counter}.xyz"):
            counter += 1
        XYZ(target_molecule).write_file(f"/jbod/rokubo/data/diffusion_model/xyz/k2en1th3/{mol_id}/target_{counter}.xyz")
        XYZ(generated_molecule).write_file(f"/jbod/rokubo/data/diffusion_model/xyz/k2en1th3/{mol_id}/generated_{counter}.xyz")
    except:
        print(f"Error processing graph with id: {target_graph_list[i].id}")
